# 02 - Training All Uncertainty Strategies

This notebook trains DenseNet-121 on CheXpert under six uncertainty-handling strategies:

1. **U-Ignore** -- mask uncertain labels out of the loss
2. **U-Zeroes** -- treat uncertain as negative
3. **U-Ones** -- treat uncertain as positive
4. **U-SelfTrained** -- two-pass: train teacher with U-Ignore, relabel uncertain, retrain
5. **U-MultiClass** -- 3-class output per pathology (neg / pos / uncertain)
6. **Label Smoothing** -- soft targets for uncertain labels

All checkpoints and results are saved to Google Drive.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab.')

In [ ]:
# Install dependencies (Colab already has torch / pandas / PIL)
!pip install -q pyyaml tqdm scikit-learn

# ---------------------------------------------------------------
# Colab data-loading optimizations
# ---------------------------------------------------------------
# Reading 200k images directly from Google Drive is painfully slow
# (Drive is rate-limited). We copy the CheXpert-v1.0-small zip /
# folder to Colab's fast local disk /content once per session.
#
# Expected layout on Drive:
#   MyDrive/
#     CheXpert-v1.0-small.zip           <-- preferred (a single file copy)
#   or
#     CheXpert-v1.0-small/              <-- a folder (slower to copy)
# ---------------------------------------------------------------
import os, shutil, time, zipfile

LOCAL_DATA_ROOT = "/content/data"                 # fast local SSD
LOCAL_CHEXPERT  = f"{LOCAL_DATA_ROOT}/CheXpert-v1.0-small"

DRIVE_ZIP    = "/content/drive/MyDrive/CheXpert-v1.0-small.zip"
DRIVE_FOLDER = "/content/drive/MyDrive/CheXpert-v1.0-small"

if IN_COLAB and not os.path.exists(LOCAL_CHEXPERT):
    os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
    t0 = time.time()
    if os.path.exists(DRIVE_ZIP):
        print(f"Unzipping {DRIVE_ZIP} -> {LOCAL_DATA_ROOT} ...")
        with zipfile.ZipFile(DRIVE_ZIP) as zf:
            zf.extractall(LOCAL_DATA_ROOT)
    elif os.path.exists(DRIVE_FOLDER):
        print(f"Copying {DRIVE_FOLDER} -> {LOCAL_CHEXPERT} (slow, ~15-30 min)...")
        shutil.copytree(DRIVE_FOLDER, LOCAL_CHEXPERT)
    else:
        raise FileNotFoundError(
            "Could not find CheXpert on Drive. Upload either "
            f"{DRIVE_ZIP} or {DRIVE_FOLDER}."
        )
    print(f"Done in {(time.time()-t0)/60:.1f} min.")

# Sanity check
if IN_COLAB:
    assert os.path.exists(f"{LOCAL_CHEXPERT}/train.csv"), "train.csv missing!"
    print("Local data ready:", LOCAL_CHEXPERT)

In [ ]:
import os
import sys
import copy
import pickle

import torch
import numpy as np

if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/Research Project'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_config, set_seed
from src.dataset import CheXpertDataset, get_train_transforms, get_val_transforms
from src.train import train_strategy, train_self_trained

config = load_config(os.path.join(PROJECT_ROOT, 'configs', 'default.yaml'))
set_seed(config['seed'])

# ---------------------------------------------------------------
# Colab overrides: point at fast local data, not Drive
# ---------------------------------------------------------------
if IN_COLAB:
    config['data']['data_dir']   = LOCAL_CHEXPERT
    config['data']['num_workers'] = 2     # Colab has ~2 usable CPU cores

# ---------------------------------------------------------------
# DEV MODE: set to a small fraction (e.g. 0.05 = 5%) while debugging,
# then set to None for the full run.
# ---------------------------------------------------------------
SUBSET_FRAC = 0.05   # <-- change to None when you're ready for a full run

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'Subset fraction: {SUBSET_FRAC}')

## Load Dataset

In [ ]:
DATA_DIR = config['data']['data_dir']
IMG_SIZE = config['data']['image_size']
TARGET_LABELS = config['data']['target_labels']

train_csv = os.path.join(DATA_DIR, config['data']['train_csv'])
valid_csv = os.path.join(DATA_DIR, config['data']['valid_csv'])

train_dataset = CheXpertDataset(
    csv_path=train_csv,
    data_dir=os.path.dirname(DATA_DIR),
    target_labels=TARGET_LABELS,
    transform=get_train_transforms(IMG_SIZE),
    subset_frac=SUBSET_FRAC,       # <-- trains on a small slice while debugging
)

# Validation set is tiny (~200 images) so we always use all of it.
val_dataset = CheXpertDataset(
    csv_path=valid_csv,
    data_dir=os.path.dirname(DATA_DIR),
    target_labels=TARGET_LABELS,
    transform=get_val_transforms(IMG_SIZE),
)

print(f'Train samples: {len(train_dataset):,}')
print(f'Valid samples: {len(val_dataset):,}')

In [ ]:
from torch.utils.data import DataLoader

BS = config['training']['batch_size']
NW = config['data']['num_workers']

val_loader = DataLoader(
    val_dataset, batch_size=BS, shuffle=False,
    num_workers=NW, pin_memory=True,
)

CKPT_DIR = config['checkpointing']['save_dir']
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')

## Train Each Strategy

Each strategy reuses the same dataset but applies different label mappings.
The `train_strategy()` function handles everything internally.

**Tip:** If Colab disconnects, you can restart from the strategy you left off on. Comment out completed strategies and re-run.

In [ ]:
all_results = []

STRATEGIES_BINARY = ['u_ignore', 'u_zeroes', 'u_ones', 'label_smoothing']

In [ ]:
# --- Binary strategies (U-Ignore, U-Zeroes, U-Ones, Label Smoothing) ---

for strat in STRATEGIES_BINARY:
    set_seed(config['seed'])

    train_loader = DataLoader(
        train_dataset, batch_size=BS, shuffle=True,
        num_workers=NW, pin_memory=True,
    )

    result = train_strategy(
        strategy=strat,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        target_labels=TARGET_LABELS,
        checkpoint_dir=CKPT_DIR,
    )
    all_results.append(result)

In [ ]:
# --- U-MultiClass ---

set_seed(config['seed'])

train_loader_mc = DataLoader(
    train_dataset, batch_size=BS, shuffle=True,
    num_workers=NW, pin_memory=True,
)

result_mc = train_strategy(
    strategy='u_multiclass',
    train_loader=train_loader_mc,
    val_loader=val_loader,
    config=config,
    device=device,
    target_labels=TARGET_LABELS,
    checkpoint_dir=CKPT_DIR,
)
all_results.append(result_mc)

In [ ]:
# --- U-SelfTrained (two-pass) ---
# This modifies train_dataset labels in-place, so run it last.

set_seed(config['seed'])

# Make a deep copy of the dataset to avoid polluting labels for other strategies
train_dataset_st = copy.deepcopy(train_dataset)

result_st = train_self_trained(
    train_dataset=train_dataset_st,
    val_loader=val_loader,
    config=config,
    device=device,
    target_labels=TARGET_LABELS,
    checkpoint_dir=CKPT_DIR,
)
all_results.append(result_st)

## Save Results

In [ ]:
results_path = os.path.join(CKPT_DIR, 'all_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(all_results, f)
print(f'Saved {len(all_results)} strategy results to {results_path}')

## Quick Summary

In [ ]:
from src.evaluate import build_summary_table

print(build_summary_table(all_results, TARGET_LABELS))